# limiteddepkit quickstart

This notebook is a package-scoped tour of `limiteddepkit`: binary response, ordinal response, count models, and post-estimation checks. It uses deterministic simulated data so the notebook is safe to run on Kaggle or Google Colab without external datasets or credentials.

What this notebook is: a reproducible adoption/demo artifact.  
What it is not: a Stata/R parity certificate or a replacement for the package validation harness.

## Install

Kaggle and Colab runtimes are usually clean. If you are running from a local checkout, skip this cell and make sure the source tree is on `PYTHONPATH`.

In [ ]:
%pip install -q "limiteddepkit[outputhub]"

## Shared imports and deterministic data helpers

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit, softmax

from limiteddepkit import (
    BinaryLogit,
    BinaryProbit,
    FirthBinaryLogit,
    IntervalRegression,
    NegativeBinomial,
    OrderedLogit,
    OrderedProbit,
    PoissonRegressor,
    Tobit,
    add_to_outputhub,
)
from limiteddepkit.experimental import MultinomialLogit, WeibullDuration
from limiteddepkit.ml.split import StratifiedKFold
from limiteddepkit.ml.validation import cross_validate

SEED = 20260730
rng = np.random.default_rng(SEED)
print("limiteddepkit import OK")

## 1. Binary response: logit and probit

A binary-response model is the canonical limited-dependent-variable starting point. Here the latent index has two covariates and a constant. We fit logit and probit to the same design matrix, then compare coefficient direction, predicted probabilities, and average marginal effects.

In [ ]:
n = 700
X_binary = pd.DataFrame({
    "const": 1.0,
    "x1": rng.normal(size=n),
    "x2": rng.normal(size=n),
})
beta_binary = np.array([-0.35, 0.85, -0.45])
y_binary = rng.binomial(1, expit(X_binary.to_numpy() @ beta_binary))

logit = BinaryLogit().fit(X_binary, y_binary)
probit = BinaryProbit().fit(X_binary, y_binary)

binary_summary = pd.concat(
    {
        "logit": logit.summary_frame()[["coef", "std_err"]],
        "probit": probit.summary_frame()[["coef", "std_err"]],
    },
    axis=1,
)

print("logit converged:", logit.converged)
print("probit converged:", probit.converged)
display(binary_summary.round(4))

display(logit.predict_proba(X_binary.iloc[:8]).round(4))
display(logit.average_marginal_effects(X_binary).round(4).to_frame("logit_AME"))

assert logit.converged and probit.converged
assert logit.params["x1"] > 0 and logit.params["x2"] < 0
assert np.allclose(logit.predict_proba(X_binary.iloc[:20]).sum(axis=1), 1.0)

## 2. Ordinal response: ordered logit and ordered probit

Ordered models are useful when the outcome has a meaningful rank, such as low/medium/high adoption or ordered survey responses. The notebook checks that thresholds are ordered and predicted category probabilities sum to one.

In [ ]:
n_ord = 900
X_ord = pd.DataFrame({
    "x1": rng.normal(size=n_ord),
    "x2": rng.normal(size=n_ord),
})
beta_ord = np.array([0.9, -0.6])
thresholds = np.array([-0.8, 0.7])
latent = X_ord.to_numpy() @ beta_ord
cum = expit(thresholds[None, :] - latent[:, None])
prob_ord = np.column_stack([cum[:, 0], cum[:, 1] - cum[:, 0], 1.0 - cum[:, 1]])
y_ord = np.array([rng.choice(3, p=row) for row in prob_ord])

ologit = OrderedLogit().fit(X_ord, y_ord)
oprobit = OrderedProbit().fit(X_ord, y_ord)

ordinal_compare = pd.DataFrame({
    "ordered_logit": ologit.params,
    "ordered_probit": oprobit.params,
})

print("ordered logit converged:", ologit.converged)
print("ordered probit converged:", oprobit.converged)
display(ordinal_compare.round(4))
display(ologit.predict_proba(X_ord.iloc[:8]).round(4))
display(ologit.average_marginal_effects(X_ord).round(4))

assert ologit.converged and oprobit.converged
assert np.all(np.diff(ologit.thresholds.to_numpy()) > 0)
assert np.allclose(ologit.predict_proba(X_ord.iloc[:25]).sum(axis=1), 1.0)

## 3. Universal Output Hub export

`limiteddepkit` includes an OutputHub adapter for ordinal-family results. This cell adds the ordered-logit model and average marginal effects table to an `OutputHub` report object. The notebook keeps the export in memory for portability on Kaggle/Colab, but the same object can be rendered or exported by downstream OutputHub workflows.

In [ ]:
try:
    from universal_output_hub import OutputHub

    hub = OutputHub("limiteddepkit ordinal report")
    hub_model = add_to_outputhub(
        hub,
        ologit,
        name="Ordered logit adoption score",
        depvar="adoption_score",
        X=X_ord,
    )

    print("OutputHub models:", len(hub.models))
    print("OutputHub tables:", len(hub.tables))
    display(hub_model.params.round(4).to_frame("coef"))
    display(hub.tables[0].data.head().round(4))
except ImportError as exc:
    print("Install the outputhub extra to run this cell:")
    print('  %pip install "limiteddepkit[outputhub]"')
    print(exc)

## 4. Count outcomes: Poisson and negative binomial

Count models are useful for non-negative integer outcomes such as events, visits, citations, claims, or failures. The negative binomial model adds dispersion beyond Poisson.

In [ ]:
n_count = 650
X_count = pd.DataFrame({
    "const": 1.0,
    "x": rng.normal(size=n_count),
})
offset = pd.Series(rng.normal(scale=0.10, size=n_count))
exposure = pd.Series(rng.uniform(0.5, 2.0, size=n_count))
mean = np.exp(X_count.to_numpy() @ np.array([-0.15, 0.45]) + offset) * exposure

y_pois = pd.Series(rng.poisson(mean))
alpha = 0.70
y_nb = pd.Series(rng.negative_binomial(1.0 / alpha, 1.0 / (1.0 + alpha * mean)))

pois = PoissonRegressor().fit(X_count, y_pois, offset=offset, exposure=exposure)
nb = NegativeBinomial().fit(X_count, y_nb, offset=offset, exposure=exposure)

poisson_summary = pois.summary_frame().reset_index(names="term")
poisson_summary.insert(0, "model", "Poisson")
nb_summary = nb.summary_frame().reset_index(names="term")
nb_summary.insert(0, "model", "Negative binomial")
count_summary = pd.concat([poisson_summary, nb_summary], ignore_index=True)

# `log_alpha` is the negative-binomial dispersion parameter. It is not a
# Poisson parameter, so the table is long-format to avoid misleading NaNs.
display(count_summary[["model", "term", "coef", "std_err", "p_value"]].round(4))
display(pd.DataFrame({
    "poisson_mean_prediction": pois.predict(X_count.iloc[:8], offset=offset.iloc[:8], exposure=exposure.iloc[:8]),
    "nb_mean_prediction": nb.predict(X_count.iloc[:8], offset=offset.iloc[:8], exposure=exposure.iloc[:8]),
}).round(4))

assert pois.converged and nb.converged
assert np.all(pois.predict(X_count.iloc[:20], offset=offset.iloc[:20], exposure=exposure.iloc[:20]) > 0)
assert np.all(nb.predict(X_count.iloc[:20], offset=offset.iloc[:20], exposure=exposure.iloc[:20]) > 0)

## 5. Multinomial choice

Multinomial logit handles unordered choices with more than two categories. This example keeps `outside` as the base category and checks that predicted probabilities across alternatives sum to one.

In [ ]:
n_multi = 900
X_multi = pd.DataFrame({
    "const": 1.0,
    "x1": rng.normal(size=n_multi),
    "x2": rng.normal(size=n_multi),
})
categories = ["outside", "bus", "car", "rail"]
coef_multi = np.array([
    [0.25, 0.65, -0.30],
    [-0.35, -0.45, 0.55],
    [0.10, 0.20, 0.25],
])
utilities = np.column_stack([np.zeros(n_multi), X_multi.to_numpy() @ coef_multi.T])
prob_multi = softmax(utilities, axis=1)
y_multi = np.array([rng.choice(categories, p=row) for row in prob_multi], dtype=object)

mlogit = MultinomialLogit().fit(
    X_multi,
    y_multi,
    category_order=categories,
    base_category="outside",
)

display(mlogit.summary_frame().head(10).round(4))
display(mlogit.predict_proba(X_multi.iloc[:8]).round(4))

assert mlogit.converged
assert np.allclose(mlogit.predict_proba(X_multi.iloc[:20]).sum(axis=1), 1.0)

## 6. Censoring and interval outcomes

Tobit and interval regression are useful when the observed outcome is censored or grouped rather than continuously observed. These examples show latent-mean prediction, observed-mean prediction, and interval-regression summaries.

In [ ]:
n_cens = 600
X_cens = pd.DataFrame({"const": 1.0, "x": rng.normal(size=n_cens)})
latent = 0.5 + 0.8 * X_cens["x"].to_numpy() + rng.normal(size=n_cens)
y_tobit = np.maximum(latent, 0.0)

tobit = Tobit().fit(X_cens, y_tobit)

grouped_lower = np.floor(latent * 2.0) / 2.0
grouped_upper = grouped_lower + 0.5
interval = IntervalRegression().fit(X_cens, grouped_lower, grouped_upper)

display(tobit.summary_frame().round(4))
display(pd.DataFrame({
    "latent_mean": tobit.predict(X_cens.iloc[:8], which="latent"),
    "observed_mean": tobit.predict(X_cens.iloc[:8]),
    "censoring_probability": tobit.predict(X_cens.iloc[:8], which="censoring_probability"),
}).round(4))
display(interval.summary_frame().round(4))

assert tobit.converged and interval.converged
assert tobit.n_censored == int(np.sum(y_tobit == 0.0))
assert interval.n_interval == len(X_cens)

## 7. Duration models

Duration models handle time-to-event outcomes with right censoring. This compact Weibull example reports event counts, shape, and positive duration predictions.

In [ ]:
n_dur = 450
X_dur = pd.DataFrame({"const": 1.0, "x": rng.normal(size=n_dur)})
beta_dur = np.array([0.3, -0.5])
shape_true = 1.5
scale = np.exp(X_dur.to_numpy() @ beta_dur)
true_duration = scale * (rng.exponential(size=n_dur) ** (1.0 / shape_true))
censoring_time = rng.exponential(scale=2.5, size=n_dur)
observed_duration = np.minimum(true_duration, censoring_time)
event = true_duration <= censoring_time

weibull = WeibullDuration().fit(X_dur, observed_duration, event)

display(weibull.summary_frame().round(4))
print("events:", weibull.n_events, "of", weibull.nobs)
print("shape:", round(float(weibull.shape_param), 4))
display(weibull.predict(X_dur.iloc[:8]).round(4).to_frame("predicted_duration"))

assert weibull.converged
assert weibull.n_events == int(event.sum())
assert np.all(weibull.predict(X_dur.iloc[:20]).to_numpy() > 0.0)

## 8. Small-sample and separation-safe binary logit

Firth binary logit is useful when ordinary binary logit faces separation or very small samples. This tiny example is intentionally separated.

In [ ]:
X_sep = pd.DataFrame(
    {"const": np.ones(9), "x": np.r_[np.zeros(4), np.ones(5)]},
    index=pd.Index(range(100, 109), name="case"),
)
y_sep = np.r_[np.zeros(4), np.ones(5)]

firth = FirthBinaryLogit().fit(X_sep, y_sep)

display(firth.summary_frame().round(4))
display(firth.conf_int(method="wald").round(4))
assert firth.converged
assert np.isfinite(firth.params.to_numpy()).all()

## 9. Outcome-aware validation layer

The workflow layer can run outcome-aware cross-validation. This example uses stratified folds for a binary-response model and reports fold-level log loss, Brier score, and accuracy.

In [ ]:
X_cv = X_binary[["x1", "x2"]].copy()
y_cv = pd.Series(y_binary, index=X_cv.index)

cv_result = cross_validate(
    lambda: BinaryLogit(),
    X_cv,
    y_cv,
    splitter=StratifiedKFold(3, shuffle=True, random_state=SEED),
    outcome="binary",
)

display(cv_result.fold_frame().round(4))
display(cv_result.summary_frame().round(4))
assert cv_result.successful_folds == 3
assert cv_result.eligible

## 10. Interpretation checklist

For a public notebook, keep the claims modest and reproducible:

- binary, multinomial, and ordinal predicted probabilities obey probability identities;
- count predictions are positive and respect offset/exposure inputs;
- `log_alpha` is shown only for negative binomial because Poisson has no dispersion parameter;
- censoring, interval, duration, and small-sample examples show dedicated limited-outcome behavior;
- convergence and inference flags are shown rather than hidden;
- parity with Stata/R belongs in the package validation harness, not in this lightweight cloud notebook.